# Lab 1 — Distance Metrics

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Learning objectives

1. Represent two loans as **numeric vectors** in feature space.
2. Compute **Euclidean**, **Manhattan**, and **cosine** distance/similarity.
3. Explain why **scaling** matters before distance-based algorithms (preview Lab 2 KNN).
4. Compare how each metric reacts when you change the loan pair.

> **Checkpoints:** cosine similarity ≈ **0.90** · Euclidean ≈ **69370** · Manhattan ≈ **71367**



## Three ways to measure "closeness"

KNN (Lab 2) picks neighbors by distance. The choice of metric changes who counts as "near":

| Metric | Formula (vectors a, b) | Intuition |
|--------|------------------------|----------|
| **Euclidean** | $\sqrt{\sum (a_i - b_i)^2}$ | Straight-line distance |
| **Manhattan** | $\sum \|a_i - b_i\|$ | Grid / city-block distance |
| **Cosine similarity** | $\frac{a \cdot b}{\|a\|\|b\|}$ | Angle between directions (scale-free) |

Cosine **distance** = $1 - \text{similarity}$. Values near **1** similarity mean the vectors point in similar directions even if magnitudes differ.


## Linear algebra refresher

<!-- cisco-enrich-2026-06 -->

Each loan is a **vector** in ℝ⁵. Distance metrics use **norms** and **dot products**:

| Operation | NumPy | Role in this lab |
|-----------|-------|------------------|
| L2 norm | `np.linalg.norm(v)` | Euclidean distance |
| L1 norm | `np.abs(v).sum()` | Manhattan distance |
| Dot product | `np.dot(a, b)` | Cosine similarity numerator |


---

## 1. Load loans and pick two feature vectors


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-04":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "lending-club" / "lending_club_sample.csv").is_file():
            GH_ROOT = parent
            break

DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

df = pd.read_csv(GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv")
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

print(f"rows: {len(df)}")
display(df[NUMERIC_FEATURES + ["default"]].head(2))


---

## 2. Compute distances (first two loans)


In [ ]:
sample = df[NUMERIC_FEATURES].iloc[:2].to_numpy(dtype=float)
a, b = sample[0], sample[1]

print(f"point A (first loan): {a.round(2)}")
print(f"point B (second loan): {b.round(2)}")

euclidean = float(np.sqrt(np.sum((a - b) ** 2)))
manhattan = float(np.sum(np.abs(a - b)))
cosine_similarity = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
cosine_distance = 1.0 - cosine_similarity

print(f"Euclidean distance: {euclidean:.4f}")
print(f"Manhattan distance: {manhattan:.4f}")
print(f"cosine similarity: {cosine_similarity:.4f}")
print(f"cosine distance: {cosine_distance:.4f}")


### Reading the numbers

Euclidean and Manhattan distances are **large** because `annual_inc` and `installment` are on very different scales than `int_rate` and `dti`. Cosine similarity is high (~**0.90**) because the vectors share similar **direction** in scaled space — a hint that cosine can be useful when magnitude matters less than pattern.


---

## 3. Per-feature contribution to Manhattan distance


In [ ]:
contrib = pd.DataFrame(
    {
        "feature": NUMERIC_FEATURES,
        "abs_diff": np.abs(a - b).round(2),
    }
).sort_values("abs_diff", ascending=False)
display(contrib)


---

## 4. Extension — try rows 10 and 50


In [ ]:
a2 = df[NUMERIC_FEATURES].iloc[9].to_numpy(dtype=float)
b2 = df[NUMERIC_FEATURES].iloc[49].to_numpy(dtype=float)

euclidean2 = float(np.sqrt(np.sum((a2 - b2) ** 2)))
manhattan2 = float(np.sum(np.abs(a2 - b2)))
cosine_sim2 = float(np.dot(a2, b2) / (np.linalg.norm(a2) * np.linalg.norm(b2)))

compare = pd.DataFrame(
    {
        "pair": ["rows 0 & 1", "rows 10 & 50"],
        "euclidean": [euclidean, euclidean2],
        "manhattan": [manhattan, manhattan2],
        "cosine_sim": [cosine_similarity, cosine_sim2],
    }
)
display(compare.round(4))


---

## 5. Why scaling matters (preview)

Without scaling, KNN treats a $10,000 difference in `annual_inc` as far more important than a 5-point `int_rate` gap. Lab 2 wraps `StandardScaler` in the pipeline for this reason.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df[NUMERIC_FEATURES].iloc[:2])
sa, sb = scaled[0], scaled[1]

euclidean_scaled = float(np.sqrt(np.sum((sa - sb) ** 2)))
cosine_scaled = float(np.dot(sa, sb) / (np.linalg.norm(sa) * np.linalg.norm(sb)))

print(f"Euclidean (scaled): {euclidean_scaled:.4f}")
print(f"cosine similarity (scaled): {cosine_scaled:.4f}")


---

## 6. Checkpoint summary


In [ ]:
assert abs(cosine_similarity - 0.8960) < 0.01
assert abs(euclidean - 69370.1405) < 100
assert abs(manhattan - 71367.0300) < 100
print("✓ All checkpoint assertions passed")


## Extension — angle between two loan vectors

In [ ]:
# unit vectors preserve direction only
ua, ub = a / np.linalg.norm(a), b / np.linalg.norm(b)
angle_rad = np.arccos(np.clip(np.dot(ua, ub), -1.0, 1.0))
angle_deg = np.degrees(angle_rad)
print(f"angle between loans 0 and 1: {angle_deg:.1f} degrees")
print(f"cosine similarity matches cos(angle): {cosine_similarity:.4f}")


---

## Reflection questions

1. Which feature contributed most to Manhattan distance for the first pair?
2. When would cosine similarity be preferred over Euclidean distance?
3. How does scaling change the story before applying KNN?

**Previous:** [Day 03 — Classification](../day-03/README.md)  
**Next:** [Lab 2 — KNN classifier](lab02_knn_classifier.ipynb)
